# Preparación de datos

**Conjunto de datos:** Dataset 2 - Lugares de emisiones

**Nombre de archivo:** emission_permits_anom_2.json

## 0. Inicialización

Instalar ydata-profiling

In [1]:
!pip install ydata-profiling

Instalar geopy

In [2]:
!pip install geopy

Instalar folium

In [3]:
!pip install folium

Importaciones

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import seaborn as sns

import folium
import json
from tabulate import tabulate

Visualización de tablas y gráficas

In [5]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [6]:
# 1. Cargar el JSON
with open('../data/original/emission_permits_anom_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Extraer features
features = data['features']

# 3. Construir DataFrame con properties y coordenadas
df = pd.DataFrame([
    {
        **feature['properties'],
        'Latitud': feature['geometry']['coordinates'][1],
        'Longitud': feature['geometry']['coordinates'][0]
    }
    for feature in features
])

df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Cuenca,Latitud,Longitud
0,73640.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,Río Bogotá,4.703418,-74.226561
1,73788.0,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,Resguardo,None,Carbón,Caldera Horno,Río Suárez,5.318407,-73.704281
2,74314.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera Horno,Río Bogotá,4.800462,-74.210355
3,75972.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,Balsillas,None,Fuel Oil No.8,Planta de Asfalto,Río Bogotá,4.678797,-74.284112
4,78824.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,El Hato,None,Carbón,Caldera Horno,Río Bogotá,4.699590,-74.193752


**1.** Transformar IDExpediente a integer

In [7]:
df['IDExpediente'] = df['IDExpediente'].astype("Int64")

**2.** Eliminar columnas innecesarias

In [8]:
df = df.drop(columns=['Cuenca'])

**3.** Eliminar duplicados

In [9]:
df = df.drop_duplicates()

**4.** Manejar la capitalización en las variables categóricas

In [10]:
df["Vereda"] = df["Vereda"].str.strip().str.upper()
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].str.strip().str.capitalize()
df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,-74.226561
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,-73.704281
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,-74.210355
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,-74.193752


**5.** Mejorar la presentación de los (sin definir)

In [11]:
df["TipoCombustible"] = df["TipoCombustible"].replace("(sin definir)", "Sin definir")
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace("(sin definir)", "Sin definir")
df[(df["TipoCombustible"] == "Sin definir") | (df["TipoFuenteEmision"] == "Sin definir")]

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
42,138622,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.118908,-73.895435
43,139320,Seguimiento y Control,Ubate,Cundinamarca,TAUSA,RASGATÁ,None,Sin definir,Horno,5.190613,-73.879585
61,150306,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.125421,-73.904398
100,171196,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.506914,-74.150135
102,171204,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.516030,-74.149127
...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,CASCO URBANO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,12.629099,-39.893628
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,URBANO,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,11.736503,-41.043145


**6.** Estandarizar la vereda

In [12]:
df["Vereda"] = df["Vereda"].replace({
    "CASCO URBANO": "AREA URBANA",
    "URBANO": "AREA URBANA",
    "CENTRO URBANO": "AREA URBANA",
})

**7.** Estandarizar los tipos de fuente

In [13]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                            326
Horno                                  131
Caldera                                 24
Caldera horno                           14
Secadores                                7
Planta de asfalto                        6
Molino                                   3
Chimenea 1                               3
Trituradora                              2
Noaplica (área de operación)             2
Reactor                                  1
Planta de asfalto adm                    1
Barrilado de grafito                     1
Filtro molino pendular                   1
Aspiración molino danioni i              1
Triturador de escombros                  1
Campana de extracción  plomo 1           1
Horno de secado                          1
Horno arcillas de soacha tipo túnel      1
Triturador de material                   1
Horno túnel 1 soacha 2                   1
Chimenea triunfo central                 1
700 bhp vr2                         

In [14]:
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace({
    "Chimenea 1": "Chimenea",
    "Noaplica (área de operación)": "No aplica (área de operación)",
    "Planta de asfalto adm": "Planta de asfalto",
    "Barrilado de grafito": "Horno",
    "Filtro molino pendular": "Molino",
    "Aspiración molino danioni i": "Molino",
    "Triturador de escombros": "Trituradora",
    "Campana de extracción  plomo 1": "Horno",
    "Horno de secado": "Horno",
    "Horno arcillas de soacha tipo túnel": "Horno",
    "Triturador de material": "Trituradora",
    "Horno túnel 1 soacha 2": "Horno",
    "Chimenea triunfo central": "Chimenea",
    "700 bhp vr2": "Caldera",
    "Planta de mezcla asfáltica": "Planta de asfalto",
    "Batería de coquización a": "Batería de coquización",
    "Molino buhler": "Molino",
    "Planta trituradora": "Trituradora",
    "Batería de producción de coque": "Batería de coquización",
})

In [15]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                      326
Horno                            136
Caldera                           25
Caldera horno                     14
Planta de asfalto                  8
Secadores                          7
Molino                             6
Trituradora                        5
Chimenea                           4
No aplica (área de operación)      2
Batería de coquización             2
Reactor                            1
Name: count, dtype: int64

**8.** Estandarizar los tipos de combustible

In [16]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      327
Carbón           128
Otros             26
Gas               20
ACPM              13
NoAplica           7
Fuel Oil No.8      5
Coque              5
Mezcla             2
Madera             1
Leña               1
Hulla              1
Name: count, dtype: int64

In [17]:
df["TipoCombustible"] = df["TipoCombustible"].replace("NoAplica", "No aplica")

In [18]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      327
Carbón           128
Otros             26
Gas               20
ACPM              13
No aplica          7
Fuel Oil No.8      5
Coque              5
Mezcla             2
Madera             1
Leña               1
Hulla              1
Name: count, dtype: int64

**9.** Estandarizar los municipios

In [19]:
df["Municipio"] = df["Municipio"].replace("LOC.USAQUEN CERROS ORIENTALES", "LOCALIDAD DE USAQUEN")

In [20]:
df["Municipio"] = df["Municipio"].str.replace("LOCALIDAD DE ", "")

In [21]:
print_table(pd.DataFrame(df["Municipio"].value_counts()))

┌─────────────────────┬─────────┐
│ Municipio           │   count │
├─────────────────────┼─────────┤
│ SOACHA              │      56 │
│ CIUDAD BOLIVAR      │      52 │
│ NEMOCON             │      39 │
│ COGUA               │      33 │
│ GIRARDOT            │      25 │
│ CUCUNUBA            │      23 │
│ TAUSA               │      18 │
│ GUACHETA            │      18 │
│ SUTATAUSA           │      15 │
│ RAQUIRA             │      15 │
│ MOSQUERA            │      14 │
│ SIBATE              │      14 │
│ LENGUAZAQUE         │      14 │
│ CAJICA              │      13 │
│ TOCANCIPA           │      12 │
│ FUSAGASUGA          │      12 │
│ VILLAPINZON         │      11 │
│ ZIPAQUIRA           │      10 │
│ COTA                │      10 │
│ FACATATIVA          │       9 │
│ SIMIJACA            │       9 │
│ VILLETA             │       8 │
│ UBATE               │       7 │
│ MADRID              │       7 │
│ RICAURTE            │       6 │
│ CHIQUINQUIRA        │       6 │
│ FUNZA       

**10.** Corregir empresas con localización inválida, usando la información geográfica dada

In [22]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Crear geolocalizador ---
geolocator = Nominatim(user_agent="geo_colombia")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=2, max_retries=2, error_wait_seconds=2.0)  # para no exceder límites

# --- 2. Filtrar las filas mal ubicadas ---
df_mal_ubicacion = df[(df['Latitud'] >= 12.27) | (df['Longitud'] >= -66.50)].copy()

print(f"Fuentes con mala ubicación: {len(df_mal_ubicacion)}")

# --- 3. Función para obtener coordenadas ---
def obtener_coordenadas_fila(row):
    # Construimos una descripción geográfica más completa
    lugar_vereda = f"{row['Vereda']}, {row['Municipio']}, {row['Departamento']}, Colombia"
    lugar_municipio = f"{row['Municipio']}, {row['Departamento']}, Colombia"
    
    try:
        location = geocode(lugar_vereda)
        if location:
            return pd.Series([location.latitude, location.longitude, "Media"])
        else:
            # Si no encuentra la vereda, probar con municipio
            location = geocode(lugar_municipio)
            if location:
                if row['Vereda'] == "AREA URBANA":
                    return pd.Series([location.latitude, location.longitude, "Media"])
                else:
                    return pd.Series([location.latitude, location.longitude, "Baja"])

    except Exception as e:
        print(f"Error en {lugar_vereda}: {e}")
    
    return pd.Series([None, None, None])

# --- 4. Aplicar función ---
df_mal_ubicacion[['Latitud', 'Longitud', 'PrecisionUbicacion']] = (
    df_mal_ubicacion.apply(obtener_coordenadas_fila, axis=1)
)

df_mal_ubicacion

Fuentes con mala ubicación: 75


,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,PrecisionUbicacion
19,124002,Seguimiento y Control,Soacha,Cundinamarca,SIBATE,CHACUA,None,Otros,Reactor,4.528223,-74.231484,Media
35,136448,Seguimiento y Control,Soacha,Cundinamarca,SOACHA,AREA URBANA,None,Gas,Horno,4.582731,-74.211754,Media
38,136886,Seguimiento y Control,Soacha,Cundinamarca,SIBATE,CHACUA,None,No aplica,Molino,4.528223,-74.231484,Media
40,137218,Seguimiento y Control,Soacha,Cundinamarca,SIBATE,CHACUA,None,Carbón,Caldera,4.528223,-74.231484,Media
44,139882,Seguimiento y Control,Soacha,Cundinamarca,SOACHA,FUSUNGA,None,Carbón,Horno,4.527299,-74.193021,Media
...,...,...,...,...,...,...,...,...,...,...,...,...
536,289234,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,RESGUARDO OCCIDENTE,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.496933,-73.625257,Baja
537,289754,Sancionatorio,Gualiva,Cundinamarca,VILLETA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.011258,-74.470230,Media
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,Media
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.304638,-74.803074,Media


**Nota:** Si presenta el error `Make this Notebook Trusted to load map: File -> Trust Notebook` y usa Anaconda, use la Anaconda Prompt para dirigirse a la carpeta `preparacion` y escriba ```jupyter trust "Dataset 2 - Fuentes de emisiones.ipynb"```

In [23]:
df_mal_ubicacion.to_csv("archivos_generados/df_mal_ubicacion_corregido.csv")

In [24]:
min_lat = df_mal_ubicacion['Latitud'].min()
max_lat = df_mal_ubicacion['Latitud'].max()
min_lon = df_mal_ubicacion['Longitud'].min()
max_lon = df_mal_ubicacion['Longitud'].max()

bounds = [
    [min_lat, min_lon],  # esquina inferior izquierda x
    [min_lat, max_lon],  # inferior derecha
    [max_lat, max_lon],  # superior derecha
    [max_lat, min_lon],  # superior izquierda
]

m = folium.Map(location=[4.71, -74.07], zoom_start=5)

# Dibujar el rectángulo
folium.Polygon(
    locations=bounds,
    color='blue',
    weight=2,
    fill=True,
    fill_opacity=0.1
).add_to(m)

# Agregar marcadores de cada ubicación del dataset
for _, row in df_mal_ubicacion.iterrows():
    color = 'red' if row['PrecisionUbicacion'] == 'Baja' else 'blue'
    folium.Marker(
        location=[row['Latitud'], row['Longitud']],
        tooltip=f"{row['IDExpediente']} - {row['Municipio']}, {row['Vereda']}, Precisión: {row['PrecisionUbicacion']}",
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(m)

# Límites geográficos reales de Colombia
real_max_lat = 12.27
real_max_lon = -66.50

# Marcas adicionales para los límites geográficos de Colombia
folium.PolyLine(
    locations=[[real_max_lat, min_lon], [real_max_lat, real_max_lon]],
    color='green',
    weight=2,
    tooltip=f"Máxima latitud de Colombia terrestre: {real_max_lat}"
).add_to(m)

folium.PolyLine(
    locations=[[min_lat, real_max_lon], [real_max_lat, real_max_lon]],
    color='green',
    weight=2,
    tooltip=f"Mínima longitud de Colombia: {real_max_lon}"
).add_to(m)

m

In [25]:
# --- 5. Crear la columna con valor constante en el df original ---
df['PrecisionUbicacion'] = 'Alta'

# --- 6. Actualizar df original con los valores corregidos ---
cols_to_update = ['Latitud', 'Longitud', 'PrecisionUbicacion']

df.update(df_mal_ubicacion[cols_to_update])
df

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,PrecisionUbicacion
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,-74.226561,Alta
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,-73.704281,Alta
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,-74.210355,Alta
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112,Alta
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,-74.193752,Alta
...,...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,Media
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186,Alta
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856,Alta
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.304638,-74.803074,Media


**11.** Obtener probabilidades de incidencia de las empresas

In [26]:
fuel_matrix = pd.read_csv('../data/enriquecida/air_fuel_matrix.csv')
source_matrix = pd.read_csv('../data/enriquecida/air_source_matrix.csv')

In [27]:
# Nos aseguramos de que los nombres de columnas coincidan
fuel_matrix = fuel_matrix.rename(columns={'Tipo de combustible': 'TipoCombustible'})
source_matrix = source_matrix.rename(columns={'Tipo de fuente': 'TipoFuenteEmision'})

for var in fuel_matrix['Variable'].unique():
    # Subconjuntos para esta variable específica
    fuel_sub = (
        fuel_matrix[fuel_matrix['Variable'] == var]
        .rename(columns={
            'Probabilidad': f'Probabilidad_Fuel_{var}',
            'Ponderación': f'Ponderacion_Fuel_{var}'
        })[['TipoCombustible', f'Probabilidad_Fuel_{var}', f'Ponderacion_Fuel_{var}']]
    )
    
    src_sub = (
        source_matrix[source_matrix['Variable'] == var]
        .rename(columns={
            'Probabilidad': f'Probabilidad_Source_{var}',
            'Ponderación': f'Ponderacion_Source_{var}'
        })[['TipoFuenteEmision', f'Probabilidad_Source_{var}', f'Ponderacion_Source_{var}']]
    )
    
    # Merges sobre df
    df = df.merge(fuel_sub, on='TipoCombustible', how='left')
    df = df.merge(src_sub, on='TipoFuenteEmision', how='left')
    
    # Calcular probabilidad
    df[f'ProbabilidadIncidencia_{var}'] = (
        (df[f'Probabilidad_Fuel_{var}'] * df[f'Ponderacion_Fuel_{var}'] +
         df[f'Probabilidad_Source_{var}'] * df[f'Ponderacion_Source_{var}']) / 9
    )
    
    # Eliminar las columnas intermedias
    df = df.drop(columns=[
        f'Probabilidad_Fuel_{var}', 
        f'Ponderacion_Fuel_{var}',
        f'Probabilidad_Source_{var}', 
        f'Ponderacion_Source_{var}'
    ])

Resultado final

In [28]:
df

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,...,ProbabilidadIncidencia_NO2,ProbabilidadIncidencia_CO,ProbabilidadIncidencia_Temp,ProbabilidadIncidencia_NO,ProbabilidadIncidencia_PM10,ProbabilidadIncidencia_BP,ProbabilidadIncidencia_NOX,ProbabilidadIncidencia_O3,ProbabilidadIncidencia_RAIN,ProbabilidadIncidencia_SRAD
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,...,0.711111,0.577778,0.055556,0.711111,0.800000,0.0,0.755556,0.333333,0.055556,0.055556
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,...,0.688889,0.644444,0.055556,0.688889,0.800000,0.0,0.733333,0.266667,0.055556,0.166667
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,...,0.888889,0.644444,0.000000,0.888889,0.800000,0.0,0.933333,0.400000,0.000000,0.111111
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,...,0.777778,0.600000,0.055556,0.777778,0.844444,0.0,0.822222,0.444444,0.055556,0.166667
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,...,0.688889,0.644444,0.055556,0.688889,0.800000,0.0,0.733333,0.266667,0.055556,0.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556
532,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556
533,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556
534,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.304638,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556


In [29]:
reporte = ProfileReport(df)
reporte.to_file("archivos_generados/Reporte perfilamiento - Dataset 2 Final.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 27/27 [00:00<00:00, 108.19it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Exportar a CSV

In [30]:
df.to_csv('../data/preparada/emission_permits.csv', index=False)